# 04 — Model Training

This notebook trains the `SentimentLSTM` architecture defined in `03_model_building.ipynb`.
It documents two full training runs — a **baseline** configuration and an **experimental** two-layer
variant — and compares their results side by side.

**Prerequisites:** `02_preprocessing.ipynb` must have been run first so that the following files exist:
- `data/vocab.pkl`
- `data/train_sequences.npy`, `data/train_labels.npy`
- `data/val_sequences.npy`, `data/val_labels.npy`
- `data/test_sequences.npy`, `data/test_labels.npy`

**Notebook outline:**
1. Setup — imports, device detection, reproducibility seed
2. Dataset and DataLoaders
3. EarlyStopping and helper functions
4. Training loop
5. Baseline run (1-layer LSTM, hidden=64)
6. Experiment run (2-layer LSTM, hidden=128)
7. Comparison and conclusion

## Setup

Add the repository root to `sys.path` so that `src/` is importable both locally and in Google Colab.
We also set the random seed across Python, NumPy, and PyTorch to ensure reproducible weight
initialisation and data shuffling.

In [ ]:
import sys, os

# ── Colab vs. local ortam tespiti ────────────────────────────────────────
if os.path.exists("/content"):
    repo_root = "/content/IMDb_Sentiment_Analysis"
    if not os.path.exists(repo_root):
        os.system("git clone https://github.com/azrakarakaya1/IMDb_Sentiment_Analysis.git " + repo_root)
    os.chdir(repo_root)
else:
    os.chdir(os.path.abspath(".."))

sys.path.insert(0, os.path.abspath("."))
print(f"Working directory: {os.getcwd()}")

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.preprocessing import Vocabulary
from src.model import SentimentLSTM

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Section 1 — Load Data

We reload the pre-computed artefacts written by `02_preprocessing.ipynb`:
- `data/vocab.pkl` — the `Vocabulary` object, used here only to get `vocab_size` for the embedding layer
- The six `.npy` files — integer sequences (shape `(N, 500)`) and float labels (shape `(N,)`)

Loading from `.npy` files is much faster than re-running the preprocessing pipeline
(seconds vs. minutes), which is important when iterating on training hyperparameters.

In [ ]:
# ── Verify prerequisite files exist ──────────────────────────────────────
required_files = [
    "data/vocab.pkl",
    "data/train_sequences.npy", "data/train_labels.npy",
    "data/val_sequences.npy",   "data/val_labels.npy",
    "data/test_sequences.npy",  "data/test_labels.npy",
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing prerequisite files: {missing}\n"
        "Please run 02_preprocessing.ipynb first to generate these files."
    )

# ── Load vocabulary ───────────────────────────────────────────────────────
vocab = Vocabulary.load("data/vocab.pkl")
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE:,}")

# ── Load encoded sequences and labels ────────────────────────────────────
train_sequences = np.load("data/train_sequences.npy")
train_labels    = np.load("data/train_labels.npy")
val_sequences   = np.load("data/val_sequences.npy")
val_labels      = np.load("data/val_labels.npy")
test_sequences  = np.load("data/test_sequences.npy")
test_labels     = np.load("data/test_labels.npy")

print(f"\nLoaded arrays:")
print(f"  train_sequences : {train_sequences.shape}  dtype={train_sequences.dtype}")
print(f"  train_labels    : {train_labels.shape}  dtype={train_labels.dtype}")
print(f"  val_sequences   : {val_sequences.shape}  dtype={val_sequences.dtype}")
print(f"  val_labels      : {val_labels.shape}  dtype={val_labels.dtype}")
print(f"  test_sequences  : {test_sequences.shape}  dtype={test_sequences.dtype}")
print(f"  test_labels     : {test_labels.shape}  dtype={test_labels.dtype}")

## Section 2 — Dataset and DataLoaders

### Why wrap numpy arrays in a `Dataset`?

PyTorch's `DataLoader` expects a `Dataset` object — a class that implements `__len__` and
`__getitem__`. Wrapping our numpy arrays in a `Dataset` lets `DataLoader` handle:

- **Batching** — splitting the 40,000 training examples into mini-batches of 64
- **Shuffling** — randomising the order of batches each epoch so the model does not
  memorise the data ordering (only applied to the training set; validation and test are
  evaluated in a fixed order for reproducibility)
- **Tensor conversion** — converting numpy arrays to the correct `torch.Tensor` dtypes on the fly

### Why `batch_size=64`?

64 is a well-tested default for sequence models: small enough to fit in Colab's free GPU
memory (typically 15 GB), large enough that gradient estimates are stable and training
progresses quickly. If you encounter an out-of-memory error, reduce this to 32.

In [ ]:
class IMDbDataset(Dataset):
    """Thin wrapper around pre-encoded IMDb sequences and labels."""

    def __init__(self, sequences: np.ndarray, labels: np.ndarray) -> None:
        # Convert to tensors once at construction time; avoids repeated conversion per batch
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return self.sequences[idx], self.labels[idx]


# ── Hyperparameters ───────────────────────────────────────────────────────
BATCH_SIZE = 64

# ── Wrap arrays in Dataset objects ───────────────────────────────────────
train_dataset = IMDbDataset(train_sequences, train_labels)
val_dataset   = IMDbDataset(val_sequences,   val_labels)
test_dataset  = IMDbDataset(test_sequences,  test_labels)

# ── Create DataLoaders ────────────────────────────────────────────────────
# shuffle=True only for training so every epoch sees examples in a different order
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches   : {len(train_loader):,}  ({len(train_dataset):,} examples)")
print(f"Val batches     : {len(val_loader):,}   ({len(val_dataset):,} examples)")
print(f"Test batches    : {len(test_loader):,}   ({len(test_dataset):,} examples)")

# Verify a single batch has the expected shapes
sample_seq, sample_lbl = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  sequences : {sample_seq.shape}  dtype={sample_seq.dtype}")
print(f"  labels    : {sample_lbl.shape}  dtype={sample_lbl.dtype}")

## Section 3 — EarlyStopping and Helper Functions

### Why early stopping?

A common failure mode when training neural networks is **overfitting**: the model learns to
memorise the training data rather than generalise to new reviews. As overfitting worsens,
training loss continues to fall while validation loss *increases* — the two curves diverge.

Early stopping monitors validation loss after each epoch. If it fails to improve for
`patience=3` consecutive epochs, training stops and the best-performing checkpoint
(lowest validation loss) is used for evaluation. This prevents the model from wasting
compute on epochs that only hurt generalisation.

### Why `patience=3`?

A patience of 3 gives the model enough room to escape temporary plateaus (where validation
loss stalls for an epoch or two before improving again), while not waiting so long that we
train deep into an overfit regime.

### `train_epoch` and `eval_epoch`

Separating training and evaluation into dedicated functions keeps the main training loop
clean and makes each function independently testable. The key difference:

- `train_epoch`: calls `model.train()` (enables dropout), runs backprop and `optimizer.step()`
- `eval_epoch`: calls `model.eval()` (disables dropout), wraps the loop in `torch.no_grad()`
  to skip gradient computation (faster and uses less memory)

In [ ]:
class EarlyStopping:
    """Saves the best checkpoint and signals when validation loss has stopped improving."""

    def __init__(self, patience: int = 3, path: str = "checkpoints/best_model.pt") -> None:
        self.patience  = patience
        self.path      = path
        self.best_loss = float("inf")
        self.counter   = 0

    def __call__(self, val_loss: float, model: nn.Module) -> bool:
        """Return True if training should stop."""
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            os.makedirs(os.path.dirname(self.path), exist_ok=True)
            torch.save(model.state_dict(), self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_epoch(model, loader, optimizer, criterion, device):
    """Run one training epoch; return (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for sequences, labels in loader:
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(sequences).squeeze(1)   # (batch,)
        loss   = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += ((output >= 0.5).float() == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    """Run one evaluation pass; return (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for sequences, labels in loader:
            sequences, labels = sequences.to(device), labels.to(device)
            output = model(sequences).squeeze(1)
            loss   = criterion(output, labels)
            total_loss += loss.item() * len(labels)
            correct    += ((output >= 0.5).float() == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total


def run_training(config: dict, tag: str):
    """
    Full training run for a given configuration.

    Parameters
    ----------
    config : dict
        Keys: vocab_size, embedding_dim, hidden_size, num_layers, dropout,
              learning_rate, max_epochs, patience, checkpoint_path
    tag : str
        Label printed in progress output (e.g. "Baseline" or "Experiment").

    Returns
    -------
    dict
        Training history with keys train_loss, val_loss, train_accuracy, val_accuracy.
    """
    print(f"\n{'='*60}")
    print(f"  {tag}")
    print(f"  num_layers={config['num_layers']}  hidden_size={config['hidden_size']}  "
          f"lr={config['learning_rate']}  patience={config['patience']}")
    print(f"{'='*60}")

    model = SentimentLSTM(
        vocab_size    = config["vocab_size"],
        embedding_dim = config["embedding_dim"],
        hidden_size   = config["hidden_size"],
        num_layers    = config["num_layers"],
        dropout       = config["dropout"],
    ).to(device)

    optimizer      = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    criterion      = nn.BCELoss()
    early_stopping = EarlyStopping(patience=config["patience"],
                                   path=config["checkpoint_path"])

    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}

    for epoch in range(1, config["max_epochs"] + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc   = eval_epoch(model, val_loader,   criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        print(f"Epoch {epoch:2d}/{config['max_epochs']}  "
              f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if early_stopping(val_loss, model):
            print(f"Early stopping triggered at epoch {epoch} "
                  f"(no improvement for {config['patience']} consecutive epochs).")
            print(f"Best val_loss: {early_stopping.best_loss:.4f}")
            break

    print(f"\nCheckpoint saved to: {config['checkpoint_path']}")
    return history


print("Helper classes and functions defined.")

## Section 4 — Plotting Helper

A single reusable function to draw the four training curves on one figure.
Plotting both loss and accuracy together on the same figure makes it easy to spot:

- **Overfitting**: train loss keeps falling, val loss diverges upward
- **Underfitting**: both losses remain high and plateau early
- **Healthy convergence**: both losses fall together and stabilise

In [ ]:
def plot_history(history: dict, title: str) -> None:
    """Plot training and validation loss/accuracy curves."""
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # Loss subplot
    ax1.plot(epochs, history["train_loss"], "b-o", label="Train Loss")
    ax1.plot(epochs, history["val_loss"],   "r-o", label="Val Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("BCE Loss")
    ax1.set_title("Loss Curves")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy subplot
    ax2.plot(epochs, [a * 100 for a in history["train_accuracy"]], "b-o", label="Train Accuracy")
    ax2.plot(epochs, [a * 100 for a in history["val_accuracy"]],   "r-o", label="Val Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Accuracy Curves")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


print("Plotting helper defined.")

## Section 5 — Baseline Training

### Configuration

| Hyperparameter | Value | Rationale |
|---|---|---|
| `vocab_size` | 20,002 | Matches vocabulary built in `02_preprocessing.ipynb` |
| `embedding_dim` | 128 | Standard starting point; good balance of capacity vs. speed |
| `hidden_size` | 64 | Sufficient for binary classification; keeps model small |
| `num_layers` | 1 | Baseline — single LSTM layer |
| `dropout` | 0.5 | Standard LSTM regularisation; prevents co-adaptation of units |
| `learning_rate` | 1e-3 | Adam default; well-tested for this task |
| `batch_size` | 64 | Fits in Colab GPU memory; stable gradient estimates |
| `max_epochs` | 20 | Upper bound; early stopping fires before this in practice |
| `patience` | 3 | Allows 3 non-improving epochs before stopping |

### What to expect

Published results for single-layer LSTM classifiers on IMDb typically land in the
**85–90% accuracy** range. We expect training to converge in roughly 5–8 epochs before
early stopping fires.

In [ ]:
baseline_config = {
    "vocab_size"      : VOCAB_SIZE,
    "embedding_dim"   : 128,
    "hidden_size"     : 64,
    "num_layers"      : 1,
    "dropout"         : 0.5,
    "learning_rate"   : 1e-3,
    "max_epochs"      : 20,
    "patience"        : 3,
    "checkpoint_path" : "checkpoints/best_model_baseline.pt",
}

baseline_history = run_training(baseline_config, tag="Baseline  (1-layer LSTM, hidden=64)")

### Baseline Results — Training History Table

In [ ]:
import pandas as pd

def history_table(history: dict) -> pd.DataFrame:
    epochs = list(range(1, len(history["train_loss"]) + 1))
    return pd.DataFrame({
        "Epoch"         : epochs,
        "Train Loss"    : [f"{v:.4f}" for v in history["train_loss"]],
        "Val Loss"      : [f"{v:.4f}" for v in history["val_loss"]],
        "Train Acc (%)" : [f"{v*100:.2f}" for v in history["train_accuracy"]],
        "Val Acc (%)"   : [f"{v*100:.2f}" for v in history["val_accuracy"]],
    })

print("Baseline training history:")
history_table(baseline_history)

### Baseline Results — Training Curves

In [ ]:
plot_history(baseline_history, "Baseline: 1-Layer LSTM (hidden=64)")

### Interpreting the Baseline Training Curves

*(Fill in after training completes.)*

**Key observations to address:**

- **Loss convergence**: Do training and validation loss decrease together, or do they diverge?
  Divergence (train continues falling while val plateaus or rises) is a sign of overfitting.
- **Accuracy gap**: Is there a large gap between training and validation accuracy?
  A gap > 5–10% typically indicates the model has memorised training patterns.
- **Early stopping**: How many epochs ran before early stopping fired?
  Stopping early (epoch 4–6) suggests the model learns quickly on this dataset.
- **Final val accuracy**: Does it fall in the expected 85–90% range?

**Typical pattern for this architecture:** The loss curves should show healthy convergence —
both falling together for the first few epochs — with a modest gap between training and
validation accuracy (< 5%), indicating the dropout layer is providing adequate regularisation.

## Section 6 — Experiment: 2-Layer LSTM

### Hypothesis

Adding a second LSTM layer and increasing the hidden size from 64 to 128 should give the
model a greater capacity to learn hierarchical patterns in text — the first layer learning
local token-level patterns, the second learning longer-range compositional structure.

**Expected outcome:** A 1–3% improvement in validation accuracy, at the cost of roughly
twice as many LSTM parameters and slower training.

**Risk:** With more parameters and the same dataset size, the model may overfit more
aggressively, which would cause validation loss to diverge earlier. If this happens,
early stopping should catch it before it gets too severe.

### Changes from baseline

| Hyperparameter | Baseline | Experiment | Reason |
|---|---|---|---|
| `num_layers` | 1 | 2 | Add a stacked LSTM layer |
| `hidden_size` | 64 | 128 | Pair larger capacity with the extra layer |

All other hyperparameters remain unchanged.

In [ ]:
experiment_config = {
    "vocab_size"      : VOCAB_SIZE,
    "embedding_dim"   : 128,
    "hidden_size"     : 128,    # increased from 64
    "num_layers"      : 2,      # added second LSTM layer
    "dropout"         : 0.5,
    "learning_rate"   : 1e-3,
    "max_epochs"      : 20,
    "patience"        : 3,
    "checkpoint_path" : "checkpoints/best_model_experiment.pt",
}

experiment_history = run_training(experiment_config, tag="Experiment (2-layer LSTM, hidden=128)")

### Experiment Results — Training History Table

In [ ]:
print("Experiment training history:")
history_table(experiment_history)

### Experiment Results — Training Curves

In [ ]:
plot_history(experiment_history, "Experiment: 2-Layer LSTM (hidden=128)")

## Section 7 — Comparison and Conclusion

We evaluate both checkpoints on the **validation** set here (the test set is held back
for `05_evaluation.ipynb`) and summarise the results in a side-by-side table.

The checkpoint with the lower validation loss is then **copied to `checkpoints/best_model.pt`**
so that `05_evaluation.ipynb` always loads the best-performing configuration.

In [ ]:
import shutil

criterion = nn.BCELoss()

def evaluate_checkpoint(checkpoint_path: str, config: dict) -> dict:
    """Load checkpoint, evaluate on val and test sets, return metrics dict."""
    model = SentimentLSTM(
        vocab_size    = config["vocab_size"],
        embedding_dim = config["embedding_dim"],
        hidden_size   = config["hidden_size"],
        num_layers    = config["num_layers"],
        dropout       = config["dropout"],
    ).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    val_loss,  val_acc  = eval_epoch(model, val_loader,  criterion, device)
    test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
    return {
        "val_loss"  : val_loss,
        "val_acc"   : val_acc,
        "test_loss" : test_loss,
        "test_acc"  : test_acc,
    }


baseline_metrics   = evaluate_checkpoint(baseline_config["checkpoint_path"],   baseline_config)
experiment_metrics = evaluate_checkpoint(experiment_config["checkpoint_path"], experiment_config)

# ── Side-by-side comparison table ────────────────────────────────────────
comparison = pd.DataFrame({
    "Configuration"  : ["Baseline (1-layer, hidden=64)", "Experiment (2-layer, hidden=128)"],
    "Val Loss"       : [f"{baseline_metrics['val_loss']:.4f}",
                        f"{experiment_metrics['val_loss']:.4f}"],
    "Val Accuracy"   : [f"{baseline_metrics['val_acc']*100:.2f}%",
                        f"{experiment_metrics['val_acc']*100:.2f}%"],
    "Test Accuracy"  : [f"{baseline_metrics['test_acc']*100:.2f}%",
                        f"{experiment_metrics['test_acc']*100:.2f}%"],
})
print("\nConfiguration comparison:")
print(comparison.to_string(index=False))

In [ ]:
# ── Select the best checkpoint and copy it to checkpoints/best_model.pt ──
if baseline_metrics["val_loss"] <= experiment_metrics["val_loss"]:
    best_source = baseline_config["checkpoint_path"]
    best_label  = "Baseline (1-layer, hidden=64)"
else:
    best_source = experiment_config["checkpoint_path"]
    best_label  = "Experiment (2-layer, hidden=128)"

shutil.copy(best_source, "checkpoints/best_model.pt")
print(f"Best configuration : {best_label}")
print(f"Checkpoint copied  : {best_source} → checkpoints/best_model.pt")
print(f"\nThis checkpoint will be loaded by 05_evaluation.ipynb.")

### Conclusion

*(Fill in after training completes.)*

**Points to address:**

1. **Did the experiment improve over the baseline?** Compare val accuracy and test accuracy.
   A difference of < 0.5% is within noise and does not justify the added complexity.

2. **Did the 2-layer model overfit more?** Look at whether the gap between train and val
   accuracy is larger in the experiment. If yes, this confirms the hypothesis risk.

3. **Training speed**: The 2-layer model should take roughly 1.5–2× longer per epoch.
   Is the accuracy gain worth the cost?

4. **Next steps**: Which checkpoint was selected as `best_model.pt`?
   What would you change in a third experiment — learning rate schedule, weight decay,
   pre-trained embeddings?